In [36]:
# !pip install openai geopy pandas numpy --quiet
# !pip install yandex_geocoder
# !pip install langchain_openai

In [37]:
import sys
print(sys.executable)
!{sys.executable} -m pip install pandas numpy geopy openai yandex-geocoder langchain langchain-openai pydantic python-dotenv --quiet

c:\Users\osman\AppData\Local\Programs\Python\Python312\python.exe


In [38]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from yandex_geocoder import Client
import pandas as pd

load_dotenv(override=True)

# API Configuration
OPENAI_BASE_URL = "https://api.deepseek.com"
MODEL = "deepseek-chat"
OPENAI_API_KEY = os.getenv("DEEPSEEK_API_KEY")
YANDEX_API_KEY = os.getenv("YANDEX_API_KEY")

# Установка переменных окружения (на всякий случай)
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['OPENAI_BASE_URL'] = OPENAI_BASE_URL

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
yandex_client = Client(YANDEX_API_KEY)

# Pandas Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("[INFO] Окружение подготовлено. Используется модель: " + MODEL)

[INFO] Окружение подготовлено. Используется модель: deepseek-chat


In [39]:
BASE = "./datasets/"

def clean_cols(df):
    df.columns = (
        df.columns.str.strip().str.lower()
        .str.replace(" ", "_").str.replace("(", "").str.replace(")", "")
    )
    return df

# Load datasets
tickets = clean_cols(pd.read_csv(BASE + "tickets.csv"))
managers = clean_cols(pd.read_csv(BASE + "managers.csv"))
offices = clean_cols(pd.read_csv(BASE + "business_units.csv"))

# Parse Manager Skills
managers["skills_list"] = managers["навыки"].apply(
    lambda x: [s.strip().upper() for s in str(x).strip("[]").replace("'", "").split(",")] 
    if pd.notna(x) else []
)
managers["load"] = 0

In [40]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import Literal

# 1. Define the Schema (Replaces manual JSON logic)


class TicketAnalysis(BaseModel):
    type: Literal["Жалоба", "Смена данных", "Консультация", "Претензия", 
                  "Неработоспособность приложения", "Мошеннические действия", "Спам"] = Field(description="Категория обращения")
    tone: Literal["Позитивный", "Нейтральный", "Негативный"] = Field(description="Тон сообщения")
    priority: int = Field(ge=1, le=10, description="1-2: Инфо/Спам, 3-5: Вопросы, 6-8: Жалобы/Ошибки, 9-10: Угрозы/Убытки")
    language: str = Field(description="Язык (RU, KZ, ENG, UZ и др.)")
    summary: str = Field(description="1-2 предложения: суть + рекомендация")

# 2. Setup the LangChain Model
llm = ChatOpenAI(model=MODEL, api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
from langchain_core.output_parsers import PydanticOutputParser
parser = PydanticOutputParser(pydantic_object=TicketAnalysis)

# 3. Create Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — аналитик банка. Проанализируй обращение клиента.\n\nОтветь СТРОГО В ФОРМАТЕ JSON. Никакого текста до или после JSON.\n\n{format_instructions}"),
    ("user", "{description}")
])

# 4. Create the Chain
analysis_chain = prompt | llm | parser

def analyze_ticket_langchain(description: str) -> dict:
    if not description or str(description).strip() == "":
        return {
            "type": "Консультация", "tone": "Нейтральный", 
            "priority": 5, "language": "RU", "summary": "Описание отсутствует."
        }
    
    try:
        # Returns a Pydantic object, we convert it to a dict
        result = analysis_chain.invoke({"description": str(description), "format_instructions": parser.get_format_instructions()})
        return result.model_dump()
    except Exception as e:
        print(f"❌ LangChain Error: {e}")
        return {"type": "Консультация", "tone": "Нейтральный", "priority": 5, "language": "RU", "summary": "Ошибка анализа."}

cols_to_clear = ["type", "tone", "priority", "language", "summary"]
tickets = tickets.drop(columns=[c for c in cols_to_clear if c in tickets.columns])

print(f"[PROCESS] Начало анализа тикетов через LLM (всего к обработке: {len(tickets)})...")

ai_results = []
total_tickets = len(tickets)

for i, (_, row) in enumerate(tickets.iterrows(), 1):
    # Вызов функции анализа
    result = analyze_ticket_langchain(row.get("описание", ""))
    ai_results.append(result)
    
    # Логирование каждой 5-й строки и последней строки
    if i % 5 == 0 or i == total_tickets:
        print(f"[PROGRESS] Проанализировано {i} из {total_tickets} обращений")

ai_df = pd.DataFrame(ai_results)
tickets = pd.concat([tickets.reset_index(drop=True), ai_df.reset_index(drop=True)], axis=1)

if 'type' in tickets.columns:
    tickets.loc[tickets['type'] == 'Спам', 'priority'] = 1

print("[SUCCESS] Анализ текста завершен.")

[PROCESS] Начало анализа тикетов через LLM (всего к обработке: 31)...
[PROGRESS] Проанализировано 5 из 31 обращений
[PROGRESS] Проанализировано 10 из 31 обращений
[PROGRESS] Проанализировано 15 из 31 обращений
[PROGRESS] Проанализировано 20 из 31 обращений
[PROGRESS] Проанализировано 25 из 31 обращений
[PROGRESS] Проанализировано 30 из 31 обращений
[PROGRESS] Проанализировано 31 из 31 обращений
[SUCCESS] Анализ текста завершен.


In [41]:
import re

def clean_address(text):
    if not text or pd.isna(text): return ""
    return re.split(r'этаж|офис|кв\.|крыло|бц|бизнес-центр|зд\.', str(text), flags=re.IGNORECASE)[0].strip().rstrip(',')

def get_lat_lon_yandex(row, is_office=True):
    missing = ['nan', 'none', '', '0', '0.0']
    
    if is_office:
        city = str(row.get('офис', '???')).strip()
        address_part = clean_address(row.get('адрес', ''))
    else:
        city = str(row.get('населённый_пункт', '')).strip()
        street = str(row.get('улица', '')).strip()
        house = str(row.get('дом', '')).strip()
        address_part = f"{street} {house}" if street not in missing else ""
    
    query = f"Казахстан, {city}, {address_part}".strip(", ")

    try:
        coords = yandex_client.coordinates(query)
        return float(coords[1]), float(coords[0])
    except Exception as err1:
        try: # Fallback to city only
            coords = yandex_client.coordinates(f"Казахстан, {city}")
            return float(coords[1]), float(coords[0])
        except Exception as err2:
            print("[Error2]", err2)
            return None, None

print("[PROCESS] Запуск геокодирования адресов...")
offices[['lat', 'lon']] = offices.apply(lambda r: pd.Series(get_lat_lon_yandex(r, True)), axis=1)
tickets[['lat', 'lon']] = tickets.apply(lambda r: pd.Series(get_lat_lon_yandex(r, False)), axis=1)

# Проверка на общий успех
if tickets[['lat', 'lon']].notna().all().all():
    print("[SUCCESS] Геокодирование завершено: все координаты найдены.")
else:
    found_count = tickets['lat'].notna().sum()
    print(f"[INFO] Геокодирование завершено. Успешно: {found_count} из {len(tickets)}.")

[PROCESS] Запуск геокодирования адресов...
[Error1] Nothing found for "Казахстан, Кокшетау, пр-т Назарбаева, д.4/2" not found
[Error1] status_code=504, body=b'Service unavailable'
[Error1] Nothing found for "Казахстан, Тургень, ул. Садовая 7.0" not found
[Error1] Nothing found for "Казахстан, Красный Яр, ул. Северная 9.0" not found
[Error1] Nothing found for "Казахстан, Кокшетау, ул. Абая 91.0" not found
[Error1] Nothing found for "Казахстан, Осакаровка, ул. Центральная 10.0" not found
[Error1] Nothing found for "Казахстан, Алматы, ул. Жандосова 162.0" not found
[Error1] Nothing found for "Казахстан, Ленгер, ул. Толе би 22.0" not found
[Error1] Nothing found for "Казахстан, Усть-Каменогорск, ул. Казахстан 30.0" not found
[Error1] Nothing found for "Казахстан, Кыргауылды, ул. Молодежная 15.0" not found
[Error1] Nothing found for "Казахстан, Индербор, ул. Центральная 4.0" not found
[Error1] Nothing found for "Казахстан, Бадам, ул. Восточная 2.0" not found
[Error1] Nothing found for "Каза

In [ ]:
def log_event(message):
    """Универсальная функция для логов бэкенда"""
    print(f"[WARNING] {message}")

: 

In [ ]:
# Merge manager data with office coordinates
managers_with_geo = pd.merge(managers, offices[["офис", "lat", "lon"]], on="офис", how="left")
rr_state = {} 

def get_sorted_offices(ticket_row, office_df):
    if pd.isna(ticket_row.get("lat")) or pd.isna(ticket_row.get("lon")):
        # ФИШКА 1: 50/50 Астана/Алматы для неизвестных адресов
        return ["Астана", "Алматы"]
    
    client_coords = (ticket_row["lat"], ticket_row["lon"])
    distances = []
    
    for _, off in office_df.iterrows():
        if pd.notna(off["lat"]):
            dist = geodesic(client_coords, (off["lat"], off["lon"])).km
            distances.append((dist, off["офис"]))
            
    distances.sort(key=lambda x: x[0])
    return [office for dist, office in distances]

def get_skill_weight(manager_row):
    """
    Веса рассчитаны на основе редкости (Inverted Ratio) навыков в базе (51 чел):
    Glav: 31% -> вес 10 (самый ценный ресурс)
    VIP:  39% -> вес 9
    ENG:  65% -> вес 5
    KZ:   92% -> вес 1  (базовый навык)
    """
    weight = 0
    skills = manager_row.get("skills_list", [])
    
    if "Глав" in str(manager_row.get("должность", "")):
        weight += 10
    if "VIP" in skills:
        weight += 9
    if "ENG" in skills:
        weight += 5
    if "KZ" in skills:
        weight += 1
        
    return weight

managers_with_geo["skill_weight"] = managers_with_geo.apply(get_skill_weight, axis=1)

def find_manager_final(row):
    ticket_id = row.get("guid_клиента", "N/A")
    if str(row.get("type", "")) == "Спам":
        return None
        
    sorted_office_names = get_sorted_offices(row, offices)
    
    # Требования тикета
    client_segment = str(row.get("сегмент_клиента", "")).upper()
    needs_vip = client_segment in ["VIP", "PRIORITY"]
    needs_glav = str(row.get("type", "")) == "Смена данных"
    lang = str(row.get("language", "RU")).upper()
    needs_lang = lang in ["KZ", "ENG"]

    def filter_pool(df_pool):
        if needs_vip:
            df_pool = df_pool[df_pool["skills_list"].apply(lambda s: "VIP" in s)]
        if needs_glav:
            df_pool = df_pool[df_pool["должность"].str.contains("Глав", case=False, na=False)]
        if needs_lang:
            df_pool = df_pool[df_pool["skills_list"].apply(lambda s: lang in s)]
        return df_pool

    final_pool = pd.DataFrame()
    found_office = None
    target_office_name = sorted_office_names[0] if sorted_office_names else "Неизвестно"

    if sorted_office_names == ["Астана", "Алматы"]:
        pool = managers_with_geo[managers_with_geo["офис"].isin(["Астана", "Алматы"])].copy()
        final_pool = filter_pool(pool)
        found_office = "Астана/Алматы"
    else:
        for office_name in sorted_office_names:
            pool = managers_with_geo[managers_with_geo["офис"] == office_name].copy()
            filtered = filter_pool(pool)
            if not filtered.empty:
                final_pool = filtered
                found_office = office_name
                break

    if final_pool.empty:
        log_event(f"[ERROR] Тикет {ticket_id}: Критическая ошибка — соответствующих специалистов не найдено ни в одном офисе")
        return "Ошибка: Спецов не найдено"
    elif found_office != target_office_name and target_office_name not in ["Астана", "Алматы"]:
        log_event(f"Тикет {ticket_id}: В ближайшем офисе '{target_office_name}' нет подходящих; найден удаленно в '{found_office}'")

    # ЛОГИКА: Сначала свободные, затем - наименее "дорогие" по навыкам (бережем ресурсы)
    final_pool = final_pool.sort_values(by=["load", "skill_weight"], ascending=[True, True])

    # Round Robin среди ТОП-2
    top_candidates = final_pool.head(2)
    candidates_names = sorted(top_candidates["фио"].tolist())
    rr_key = tuple(candidates_names)
    
    current_idx = rr_state.get(rr_key, 0)
    winner_name = candidates_names[current_idx % len(candidates_names)]
    
    rr_state[rr_key] = current_idx + 1
    managers_with_geo.loc[managers_with_geo["фио"] == winner_name, "load"] += 1
    
    return winner_name

from geopy.distance import geodesic
# Запуск распределения
print("[PROCESS] Распределение тикетов по менеджерам (SLA + Гео + Балансировка)...")
tickets["assigned_manager"] = tickets.apply(find_manager_final, axis=1)
print("[SUCCESS] Распределение завершено.")


# export filename__output.csv:
with open("filename__output.csv", "w", encoding="utf-8") as f:
    tickets.to_csv(f, index=False)

[PROCESS] Распределение тикетов по менеджерам (SLA + Гео + Балансировка)...
[WARNING] Тикет 16a8690e-4aec-f011-8406-0022481bad7e: В ближайшем офисе 'Караганда' нет подходящих; найден удаленно в 'Астана'
[WARNING] Тикет 117af0c4-cb00-f111-8407-0022481ba51f: В ближайшем офисе 'Караганда' нет подходящих; найден удаленно в 'Астана'
[SUCCESS] Распределение завершено.


In [1]:
with pd.option_context('display.max_rows', None):
    display(tickets)

NameError: name 'pd' is not defined